# CP2 paired verification — server GPU lab (Beauty + ML-1M)

Versi ini disiapkan untuk server lab KC, bukan Google Colab. Notebook menjalankan baseline BSARec vanilla dan BSARec + time-bucket embedding pada source commit, data, runtime, split, serta seed yang sama. Training tidak berjalan otomatis saat file dibuka.

**Sebelum membuka notebook:** salin empat file berikut ke `~/kelompok10/cp2_gpu/data/`: `Beauty.txt`, `Beauty_cp2_time_buckets.txt`, `ML-1M.txt`, `ML-1M_cp2_time_buckets.txt`. Notebook akan membaca file dari folder itu dan menyiapkan source di `~/kelompok10/cp2_gpu/BSARec`.

Pilih satu GPU yang cukup kosong sesuai aturan lab. Jupyter harus diluncurkan dari virtual environment proyek dengan `CUDA_VISIBLE_DEVICES=<ID_GPU_FISIK>`, misalnya `CUDA_VISIBLE_DEVICES=1 jupyter lab --no-browser --ip=127.0.0.1 --port=8888`. Dalam proses yang dibatasi ke satu GPU tersebut, BSARec memakai `gpu_id=0` (indeks GPU terlihat di dalam proses). Jangan hentikan proses kelompok lain.

Urutan notebook: setup -> cek empat input -> enam baseline (dua dataset × seed 42/43/44) -> patch time-aware -> smoke test -> enam varian time-aware -> rangkuman delta. Total 12 run penuh; lakukan backup kode/notebook sebelum training dan jalankan sel satu per satu, bukan Run All.


In [ ]:
# Setup source BSARec upstream pada commit yang dipakai audit CP1.
%cd ~/kelompok10/cp2_gpu
!test -d BSARec/.git || git clone https://github.com/yehjin-shin/BSARec.git BSARec
!git -C BSARec checkout c80bdc0f1a889b78fe9a27e2058389cc84caea84
!python -m pip install -q scipy tqdm pandas
%cd ~/kelompok10/cp2_gpu/BSARec/src
import os, torch
print('PyTorch:', torch.__version__, '| CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available(), '| visible GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'Visible GPU {i}:', torch.cuda.get_device_name(i))
else:
    raise RuntimeError('CUDA tidak tersedia di kernel ini. Restart Jupyter dengan CUDA_VISIBLE_DEVICES=<GPU_ID> dari venv proyek.')


In [ ]:
# Ambil data dari folder proyek server, tanpa google.colab atau Google Drive.
from pathlib import Path
import shutil

project_dir = Path.home() / 'kelompok10' / 'cp2_gpu'
source_dir = project_dir / 'data'
repo_src = project_dir / 'BSARec' / 'src'
target_dir = repo_src / 'data'
target_dir.mkdir(parents=True, exist_ok=True)
required = [
    'Beauty.txt', 'Beauty_cp2_time_buckets.txt',
    'ML-1M.txt', 'ML-1M_cp2_time_buckets.txt',
]
missing = [name for name in required if not (source_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f'File belum ada di {source_dir}: {missing}')
for name in required:
    shutil.copy2(source_dir / name, target_dir / name)
    print(f'{source_dir / name} -> {target_dir / name}')
print('Semua input tersedia; ukuran (bytes):')
for name in required:
    print(name, (target_dir / name).stat().st_size)


## A. Baseline vanilla — jalankan sebelum patch time-aware

Sel berikut memakai source upstream tanpa patch waktu. Jalankan semua enam run ini lebih dulu. Parameter lain memakai default upstream yang sama dengan CP1; log akan mencatat Namespace lengkap.


In [ ]:
# Baseline vanilla Beauty, seed 42
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 42 --train_name BSARec_V_CP2_Beauty_s42 --num_workers 4

In [ ]:
# Baseline vanilla Beauty, seed 43
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 43 --train_name BSARec_V_CP2_Beauty_s43 --num_workers 4

In [ ]:
# Baseline vanilla Beauty, seed 44
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 44 --train_name BSARec_V_CP2_Beauty_s44 --num_workers 4

In [ ]:
# Baseline vanilla ML-1M, seed 42
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 42 --train_name BSARec_V_CP2_ML-1M_s42 --num_workers 2

In [ ]:
# Baseline vanilla ML-1M, seed 43
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 43 --train_name BSARec_V_CP2_ML-1M_s43 --num_workers 2

In [ ]:
# Baseline vanilla ML-1M, seed 44
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 44 --train_name BSARec_V_CP2_ML-1M_s44 --num_workers 2

## B. Patch time-aware dan smoke test

Jalankan patch setelah vanilla selesai. Sel patch dan smoke test sama dengan notebook CP2 pilot yang sudah dieksekusi. Smoke test harus lulus sebelum training varian waktu.


In [ ]:
%%writefile /mnt/home/probation-01/kelompok10/cp2_gpu/BSARec/src/cp2_time_runtime.py
import os
import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler
from dataset import RecDataset


def read_time_buckets(args, seq_dic):
    item_path = args.data_dir + args.data_name + '.txt'
    time_path = os.path.join(args.data_dir, args.data_name + '_cp2_time_buckets.txt')
    item_rows = []
    with open(item_path, encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            fields = line.split()
            if len(fields) < 2:
                raise ValueError(f'{item_path}:{line_no}: row kosong/pendek')
            item_rows.append((int(fields[0]), [int(v) for v in fields[1:]]))
    time_rows = []
    with open(time_path, encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            fields = line.split()
            if len(fields) < 2:
                raise ValueError(f'{time_path}:{line_no}: row kosong/pendek')
            time_rows.append((int(fields[0]), [int(v) for v in fields[1:]]))
    if len(item_rows) != len(time_rows) or len(item_rows) != len(seq_dic['user_seq']):
        raise ValueError('Jumlah baris item/time/model berbeda')
    time_sequences = []
    for i, ((item_uid, items), (time_uid, buckets), model_items) in enumerate(
        zip(item_rows, time_rows, seq_dic['user_seq']), 1
    ):
        if item_uid != time_uid:
            raise ValueError(f'User ID tidak cocok pada baris {i}: item={item_uid}, time={time_uid}')
        if len(items) != len(buckets) or len(items) != len(model_items):
            raise ValueError(f'Panjang item/bucket tidak cocok pada baris {i}')
        if any(bucket < 0 or bucket > 7 for bucket in buckets):
            raise ValueError(f'ID bucket di luar 0..7 pada baris {i}')
        time_sequences.append(buckets)
    return time_sequences


class TimeRecDataset(RecDataset):
    '''Keeps the upstream split/windows and mirrors them for time IDs.'''
    def __init__(self, args, user_seq, time_seq, data_type='train'):
        if len(user_seq) != len(time_seq):
            raise ValueError('Jumlah sequence item/time berbeda')
        self.time_user_seq = []
        self.max_len = args.max_seq_length
        super().__init__(args, user_seq, data_type=data_type)
        if data_type == 'train':
            for items, times in zip(user_seq, time_seq):
                item_prefix = items[-(self.max_len + 2):-2]
                time_prefix = times[-(self.max_len + 2):-2]
                if len(item_prefix) != len(time_prefix):
                    raise ValueError('Panjang training prefix item/time berbeda')
                for end in range(1, len(item_prefix) + 1):
                    self.time_user_seq.append(time_prefix[:end])
        elif data_type == 'valid':
            self.time_user_seq = [times[:-1] for times in time_seq]
        else:
            self.time_user_seq = [list(times) for times in time_seq]
        if len(self.time_user_seq) != len(self.user_seq):
            raise ValueError(f'Jumlah window item/time berbeda: {len(self.user_seq)} vs {len(self.time_user_seq)}')

    def __getitem__(self, index):
        base = super().__getitem__(index)
        history_times = self.time_user_seq[index][:-1]  # target-position time is excluded
        if len(history_times) > self.max_len:
            history_times = history_times[-self.max_len:]
        pad_len = self.max_len - len(history_times)
        padded_times = [0] * pad_len + history_times
        if len(padded_times) != self.max_len:
            raise AssertionError('Panjang time tensor salah')
        return (*base, torch.tensor(padded_times, dtype=torch.long))


def get_time_dataloder(args, seq_dic):
    time_seq = read_time_buckets(args, seq_dic)
    train_ds = TimeRecDataset(args, seq_dic['user_seq'], time_seq, data_type='train')
    eval_ds = TimeRecDataset(args, seq_dic['user_seq'], time_seq, data_type='valid')
    test_ds = TimeRecDataset(args, seq_dic['user_seq'], time_seq, data_type='test')
    train_loader = DataLoader(train_ds, sampler=RandomSampler(train_ds), batch_size=args.batch_size, num_workers=args.num_workers)
    eval_loader = DataLoader(eval_ds, sampler=SequentialSampler(eval_ds), batch_size=args.batch_size, num_workers=args.num_workers)
    test_loader = DataLoader(test_ds, sampler=SequentialSampler(test_ds), batch_size=args.batch_size, num_workers=args.num_workers)
    return train_loader, eval_loader, test_loader


In [ ]:
# Tambahkan bucket waktu ke dataset, trainer, dan embedding BSARec.
from pathlib import Path
import re

src = Path.home() / 'kelompok10' / 'cp2_gpu' / 'BSARec' / 'src'

def replace_once(path, old, new):
    text = path.read_text(encoding='utf-8')
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f'{path.name}: expected exactly one anchor, found {count}: {old!r}')
    path.write_text(text.replace(old, new, 1), encoding='utf-8')

# Preserve upstream split/evaluator, substitute only the loader implementation.
dataset_path = src / 'dataset.py'
dataset_text = dataset_path.read_text(encoding='utf-8')
match = re.search(r'(?ms)^def get_dataloder\(args,\s*seq_dic\):.*\Z', dataset_text)
if not match:
    raise RuntimeError('Tidak menemukan fungsi get_dataloder upstream; patch dihentikan tanpa mengubah kode.')
dataset_text = dataset_text[:match.start()] + '''def get_dataloder(args, seq_dic):
    from cp2_time_runtime import get_time_dataloder
    return get_time_dataloder(args, seq_dic)
'''
dataset_path.write_text(dataset_text, encoding='utf-8')

# Pass the parallel time tensor through the existing BSARec training/evaluation loop.
trainer_path = src / 'trainers.py'
replace_once(trainer_path,
    'user_ids, input_ids, answers, neg_answer, same_target = batch',
    'user_ids, input_ids, answers, neg_answer, same_target, time_ids = batch')
replace_once(trainer_path,
    'self.model.calculate_loss(input_ids, answers, neg_answer, same_target, user_ids)',
    'self.model.calculate_loss(input_ids, answers, neg_answer, same_target, user_ids, time_ids=time_ids)')
replace_once(trainer_path,
    'user_ids, input_ids, answers, _, _ = batch',
    'user_ids, input_ids, answers, _, _, time_ids = batch')
replace_once(trainer_path,
    'self.model.predict(input_ids, user_ids)',
    'self.model.predict(input_ids, user_ids, time_ids=time_ids)')

# Add a trainable 8-row time embedding; padded item positions get zero time contribution.
model_path = src / 'model' / 'bsarec.py'
replace_once(model_path,
    '        self.dropout = nn.Dropout(args.hidden_dropout_prob)\n',
    '        self.dropout = nn.Dropout(args.hidden_dropout_prob)\n        self.time_embeddings = nn.Embedding(8, args.hidden_size)\n')
replace_once(model_path,
    '    def forward(self, input_ids, user_ids=None, all_sequence_output=False):',
    '    def forward(self, input_ids, user_ids=None, all_sequence_output=False, time_ids=None):')
replace_once(model_path,
    '        sequence_emb = self.add_position_embedding(input_ids)',
    '        sequence_emb = self.add_time_position_embedding(input_ids, time_ids)')
helper = '''    def add_time_position_embedding(self, input_ids, time_ids):
        if time_ids is None:
            raise ValueError("CP2 BSARec requires time bucket IDs")
        seq_len = input_ids.size(1)
        position_ids = torch.arange(seq_len, dtype=torch.long, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand_as(input_ids)
        item_emb = self.item_embeddings(input_ids)
        position_emb = self.position_embeddings(position_ids)
        time_emb = self.time_embeddings(time_ids)
        time_emb = time_emb * input_ids.ne(0).unsqueeze(-1).to(time_emb.dtype)
        sequence_emb = item_emb + position_emb + time_emb
        sequence_emb = self.LayerNorm(sequence_emb)
        return self.dropout(sequence_emb)

'''
forward_anchor = '    def forward(self, input_ids, user_ids=None, all_sequence_output=False, time_ids=None):'
text = model_path.read_text(encoding='utf-8')
if text.count(forward_anchor) != 1:
    raise RuntimeError('Anchor forward BSARec tidak unik')
model_path.write_text(text.replace(forward_anchor, helper + forward_anchor, 1), encoding='utf-8')
replace_once(model_path,
    '    def calculate_loss(self, input_ids, answers, neg_answers, same_target, user_ids):',
    '    def predict(self, input_ids, user_ids=None, time_ids=None):\n        return self.forward(input_ids, user_ids, False, time_ids)\n\n    def calculate_loss(self, input_ids, answers, neg_answers, same_target, user_ids, time_ids=None):')
replace_once(model_path,
    '        seq_output = self.forward(input_ids)',
    '        seq_output = self.forward(input_ids, time_ids=time_ids)')
print('Patch CP2 diterapkan.')


In [ ]:
# Syntax check dan smoke test singkat: loader + forward/loss BSARec.
!python -m py_compile dataset.py trainers.py model/bsarec.py cp2_time_runtime.py
from types import SimpleNamespace
import torch
from dataset import get_seq_dic
from cp2_time_runtime import read_time_buckets, TimeRecDataset
from model.bsarec import BSARecModel

args = SimpleNamespace(
    data_name='Beauty', data_dir='./data/', max_seq_length=50, model_type='BSARec',
    batch_size=2, num_workers=0, hidden_size=64, hidden_dropout_prob=0.1,
    num_hidden_layers=2, num_attention_heads=1, attention_probs_dropout_prob=0.1,
    hidden_act='gelu', initializer_range=0.02, alpha=0.7, c=5,
)
seq_dic, max_item, num_users = get_seq_dic(args)
args.item_size = max_item + 1
args.num_users = num_users + 1
time_seq = read_time_buckets(args, seq_dic)
small_ds = TimeRecDataset(args, seq_dic['user_seq'][:3], time_seq[:3], data_type='test')
examples = [small_ds[i] for i in range(min(2, len(small_ds)))]
batch = [torch.stack([ex[j] for ex in examples]) for j in range(6)]
model = BSARecModel(args)
loss = model.calculate_loss(batch[1], batch[2], batch[3], batch[4], batch[0], time_ids=batch[5])
assert torch.isfinite(loss).item(), loss
assert batch[5].shape == batch[1].shape == (len(examples), 50)
print('Smoke test OK | loss finite | input/time shape:', tuple(batch[1].shape), tuple(batch[5].shape))


## C. BSARec + time embedding — paired seeds

Jalankan enam run satu per satu. Gunakan seed yang sama dengan baseline. Jangan jalankan dua training bersamaan.


In [ ]:
# Time-aware Beauty, seed 42
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 42 --train_name BSARec_T_CP2_Beauty_s42 --num_workers 4

In [ ]:
# Time-aware Beauty, seed 43
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 43 --train_name BSARec_T_CP2_Beauty_s43 --num_workers 4

In [ ]:
# Time-aware Beauty, seed 44
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 44 --train_name BSARec_T_CP2_Beauty_s44 --num_workers 4

In [ ]:
# Time-aware ML-1M, seed 42
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 42 --train_name BSARec_T_CP2_ML-1M_s42 --num_workers 2

In [ ]:
# Time-aware ML-1M, seed 43
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 43 --train_name BSARec_T_CP2_ML-1M_s43 --num_workers 2

In [ ]:
# Time-aware ML-1M, seed 44
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 44 --train_name BSARec_T_CP2_ML-1M_s44 --num_workers 2

## D. Ekstrak hasil final dan delta berpasangan

Sel ini mengambil metrik dari bagian Test Score pada log, memastikan enam pasangan tersedia, lalu menyimpan CSV. Jika ada run gagal, perbaiki/rerun hanya run itu sebelum analisis.


In [ ]:
import ast, csv, re
from pathlib import Path
import pandas as pd

METRICS=['HR@5','NDCG@5','HR@10','NDCG@10','HR@20','NDCG@20']
LOG_DIR=Path('output')

def parse_test(run_name):
    path=LOG_DIR/f'{run_name}.log'
    if not path.exists(): raise FileNotFoundError(path)
    text=path.read_text(encoding='utf-8',errors='replace')
    marker='---------------Test Score---------------'
    if marker not in text: raise ValueError(f'Test Score belum ada: {path}')
    tail=text.rsplit(marker,1)[1]
    found=[]
    for line in tail.splitlines():
        m=re.search(r"\{[^{}]+\}",line)
        if not m: continue
        try: obj=ast.literal_eval(m.group(0))
        except Exception: continue
        if isinstance(obj,dict) and all(k in obj for k in METRICS): found.append(obj)
    if not found: raise ValueError(f'Metrik test tidak terbaca: {path}')
    return {k:float(found[-1][k]) for k in METRICS}

rows=[]
for ds in ['Beauty','ML-1M']:
    for seed in [42,43,44]:
        v=parse_test(f'BSARec_V_CP2_{ds}_s{seed}')
        t=parse_test(f'BSARec_T_CP2_{ds}_s{seed}')
        for metric in METRICS:
            rows.append({'dataset':ds,'seed':seed,'metric':metric,'vanilla':v[metric],'time_aware':t[metric],'delta':t[metric]-v[metric]})
detail=pd.DataFrame(rows)
summary=(detail.groupby(['dataset','metric']).agg(mean_vanilla=('vanilla','mean'),mean_time_aware=('time_aware','mean'),mean_delta=('delta','mean'),sd_paired_delta=('delta','std'),seeds_positive=('delta',lambda x:int((x>0).sum()))).reset_index())
detail.to_csv('CP2_vanilla_vs_time_paired_by_seed.csv',index=False)
summary.to_csv('CP2_vanilla_vs_time_summary.csv',index=False)
display(detail.round(5))
display(summary.round(5))
print('Saved paired per-seed and summary CSVs.')

## E. Pemeriksaan dan batas kesimpulan

- Bandingkan Namespace di log baseline/time untuk memastikan konfigurasi sama selain fitur waktu dan nama run.
- Pastikan semua enam pasangan memakai split, target, masking, checkpoint selection, dan evaluator yang sama; source split/evaluator berasal dari commit upstream yang dipin di awal.
- Laporkan per-seed delta, mean, dan SD. Tiga seed memberi bukti awal, bukan jaminan generalisasi.
- Untuk interval kepercayaan yang lebih informatif, simpan hit/NDCG per user untuk kedua model pada test users yang sama, lalu lakukan paired bootstrap per user. Metrik agregat log saja tidak cukup untuk bootstrap itu.
- Simpan notebook executed, CSV, log, checkpoint, commit, versi PyTorch/CUDA, dan hash input. Hasil tetap deskriptif sampai eksperimen berpasangan selesai dan diperiksa.
